In [ ]:
import os
import sys
import pickle

path = os.path.abspath("")
splitted = path.split(os.sep)
user_independent = os.path.join(
    splitted[0] + os.sep, splitted[1], splitted[2], splitted[3]
)
src_path = os.path.join(user_independent, "GitHub", "Photoswitching")
sys.path.append(src_path)

import src.blinking as bl
import src.distributions as dist
import src.emissions as em
import src.fcs as fcs_p
import src.figure as fi
import src.fluorophores as fl
import src.formulas as fo
import src.miscellaneous as mi
import src.network as net
import src.simulation as si
import src.prediction as pr
import src.analysis as an
import src.transitions as tr

import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

import warnings


def custom_warning_format(msg, category, filename, lineno, line=None):
    if line is None:
        import linecache

        line = linecache.getline(filename, lineno)
    return f"WARNING for line: {line} {msg} \n"

warnings.formatwarning = custom_warning_format

### dstorm

In [321]:
def prepare_transition_set_ofret(number_fluorophores, distance):
    fluorophores = fl.construct_fluorophores(
    name="cy5_dna", distance=distance, count=number_fluorophores, shape="square"
    )
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

    transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=2.5,
    wavelength=640,
    bleaching=False,
    energy_transfer=True,
    dstorm=True,
    dstorm_parameters={'reducing_agent':'mea',
    'concentration':100,
    'ph':7.5},
    energy_transfer_parameters={'overwrite': {'off': [1, 0.0001]}, 
                                'exclude': ['s0']}
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    return transition_set


### Excitation rate calculation
2.5 kW/cm² was measured as the experimental irradiance. It is very likely that this was the average (can either be defined as for a very long time; or from pulse to pulse, including the interval in between with no laser emission). The laser pulses have a frequency of 80MHz, meaning within 1 s, there were 8e7 pulses, each with a duration of 50 ps (5e-11s). This means that during 1 s,  5e-11*8e7s = 4e-3s of irradiation happened. This means that the irradiance of a single pulse must have been 1/4e-3 times higher than the average. 

### 4F 3nm dSTORM

In [249]:
filepath = r"c:\Users\vie43sq\Desktop\Simulations\simulation_data\OET_lifetimes\4f_3nm_dstorm"
lifetimes_file = fr"{filepath}\lifetimes.npy"
event_time_series_file = fr"{filepath}\event_time_series.parquet"

transition_set = prepare_transition_set_ofret(4, 3)
rng = np.random.default_rng(42)
emis = em.Emissions(frame_time='1ms', bandpass=[665, 731], seed=rng)
Da, D, lifetimes_all = \
    emis.tcspc(transition_set, number_pulses=1e8, excitation_rates={'cy5_dna': 3.75e9},
               time_between_pulses=12.5e-9)

emis.event_time_series.to_frame().to_parquet(event_time_series_file)
np.save(lifetimes_file, lifetimes_all)

### 4F 18nm dSTORM

In [250]:
filepath = r"c:\Users\vie43sq\Desktop\Simulations\simulation_data\OET_lifetimes\4f_18nm_dstorm"
lifetimes_file = fr"{filepath}\lifetimes.npy"
event_time_series_file = fr"{filepath}\event_time_series.parquet"

transition_set_18 = prepare_transition_set_ofret(4, 18)
rng = np.random.default_rng(42)
emis_18 = em.Emissions(frame_time='1ms', bandpass=[665, 731], seed=rng)
Da_18, D_18, lifetimes_all_18 = \
    emis_18.tcspc(transition_set_18, number_pulses=1e8, excitation_rates={'cy5_dna': 3.75e9},
               time_between_pulses=12.5e-9)

emis_18.event_time_series.to_frame().to_parquet(event_time_series_file)
np.save(lifetimes_file, lifetimes_all_18)

### No dstorm

In [251]:
def prepare_transition_set_no_dstorm(number_fluorophores, distance):
    fluorophores = fl.construct_fluorophores(
    name="cy5_dna", distance=distance, count=number_fluorophores, shape="square"
    )
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

    transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=2.5,
    wavelength=640,
    bleaching=False,
    energy_transfer=True,
    dstorm=False,
    dstorm_parameters={'reducing_agent':'mea',
    'concentration':100,
    'ph':7.5},
    energy_transfer_parameters={'overwrite': {'off': [1, 0.0001]}, 
                                'exclude': ['s0']}
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    return transition_set

### 3 nm no dstorm

In [252]:
folder_path = r"c:\Users\vie43sq\Desktop\Simulations\simulation_data\OET_lifetimes\4f_3nm_no_dstorm"
lifetimes_file = fr"{folder_path}\lifetimes.npy"
event_time_series_file = fr"{folder_path}\event_time_series.parquet"

transition_set = prepare_transition_set_no_dstorm(4, 3)
rng = np.random.default_rng(42)
emis = em.Emissions(frame_time='1ms', bandpass=[665, 731], seed=rng)
Da, D, lifetimes_all = \
    emis.tcspc(transition_set, number_pulses=1e8, excitation_rates={'cy5_dna': 3.75e9},
               time_between_pulses=12.5e-9)

emis.event_time_series.to_frame().to_parquet(event_time_series_file)
np.save(lifetimes_file, lifetimes_all)

### 18 nm no dstorm

In [253]:
folder_path = r"c:\Users\vie43sq\Desktop\Simulations\simulation_data\OET_lifetimes\4f_18nm_no_dstorm"
lifetimes_file = fr"{folder_path}\lifetimes.npy"
event_time_series_file = fr"{folder_path}\event_time_series.parquet"

transition_set = prepare_transition_set_no_dstorm(4, 18)
rng = np.random.default_rng(42)
emis = em.Emissions(frame_time='1ms', bandpass=[665, 731], seed=rng)
Da, D, lifetimes_all = \
    emis.tcspc(transition_set, number_pulses=1e8, excitation_rates={'cy5_dna': 3.75e9},
               time_between_pulses=12.5e-9)

emis.event_time_series.to_frame().to_parquet(event_time_series_file)
np.save(lifetimes_file, lifetimes_all)

### 3 nm dstorm bleaching time series

In [322]:
def prepare_transition_set_ofret_bl(number_fluorophores, distance):
    fluorophores = fl.construct_fluorophores(
    name="cy5_dna", distance=distance, count=number_fluorophores, shape="square"
    )
    fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

    transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=2.5,
    wavelength=640,
    bleaching=True,
    energy_transfer=True,
    dstorm=True,
    dstorm_parameters={'reducing_agent':'mea',
    'concentration':100,
    'ph':7.5},
    energy_transfer_parameters={'overwrite': {'off': [1, 0.0001]}, 
                                'exclude': ['s0']}
    )
    transition_set = tr.TransitionSet(transitions, fluorophore_system)
    transition_set.finalize()

    return transition_set

In [ ]:
transition_set = prepare_transition_set_ofret_bl(4, 3)
rng = np.random.default_rng(42)
emis = em.Emissions(frame_time='1ms', bandpass=[665, 731], seed=rng)
lifetimes_DA, lifetimes_D, lifetimes_all, simulation_object = \
    emis.tcspc(transition_set, number_pulses=1e10, excitation_rates={'cy5_dna': 3.75e9},
            time_between_pulses=12.5e-9, details=True)

In [ ]:
folder_path = r"C:\Users\vie43sq\Desktop\Simulations\simulation_data\OET_lifetimes\4f_3nm_dstorm_bl"
lifetimes_file = fr"{folder_path}\lifetimes.npy"
time_series_file = fr"{folder_path}\time_series.npy"
transition_series_file = fr"{folder_path}\transition_series.npy"
state_series_file = fr"{folder_path}\state_series.npy"
event_time_series_file = fr"{folder_path}\event_time_series.parquet"
transition_set_file = fr"{folder_path}\transition_set.pkl"
emis.event_time_series.to_frame().to_parquet(event_time_series_file)
np.save(lifetimes_file, lifetimes_all)
np.save(time_series_file, simulation_object.time_series)
np.save(transition_series_file, simulation_object.transition_series)
np.save(state_series_file, simulation_object.state_series)
with open(transition_set_file, "wb") as f:
    pickle.dump(transition_set, f)

In [ ]:
simulation_object.time_series.size * 64 / 1000 / 1000 / 1000 / 8

0.27385212800000003